# Generar y auditar el plan de balanceo cap 200

Este notebook recupera el artifact ya existente `14770171764` (run `45038801547`), reconstruye los manifiestos fuente desde los JSONL enriquecidos del plan anterior y ejecuta el **script real del repositorio** `scripts/build_balanced_training_plan.py` con `--per-class-cap 200`.

No vuelve a indexar anotaciones, no descarga imágenes/máscaras, no entrena modelos y no requiere GPU. Guarda manifiestos, informes y comparación de caps en Google Drive.

En Colab Secrets crea `GITHUB_TOKEN` con permisos `Contents: Read-only` y `Actions: Read-only`, limitado a este repositorio. Si no existe el secreto, se solicitará de forma oculta. No pegues el token en el notebook ni en el chat.

## 1. Montar Drive

Los resultados quedarán en `MyDrive/ProcesamientoImagenes/balanceo_cap200`.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json, zipfile, hashlib, subprocess, shutil, datetime, requests

drive.mount('/content/drive')
OUT_DRIVE = Path('/content/drive/MyDrive/ProcesamientoImagenes/balanceo_cap200')
OUT_DRIVE.mkdir(parents=True, exist_ok=True)
print('Destino:', OUT_DRIVE)


## 2. Actualizar repositorio privado

Se requiere el PAT de solo lectura en Contents y Actions. La rama debe ser `urban-models-and-benchmark`.


In [ ]:
from getpass import getpass
REPO='Leohf04/Procesamiento-de-Im-genes-'
BRANCH='urban-models-and-benchmark'
WORKDIR=Path('/content/Procesamiento-de-Im-genes-')
try:
    from google.colab import userdata
    TOKEN=userdata.get('GITHUB_TOKEN')
except Exception:
    TOKEN=None
if not TOKEN:
    TOKEN=getpass('GitHub PAT (Contents: Read-only + Actions: Read-only): ')
if not TOKEN:
    raise RuntimeError('Falta el token GitHub.')
askpass=Path('/tmp/colab_git_askpass.sh')
askpass.write_text('#!/bin/sh\ncase "$1" in\n *Username*) echo x-access-token ;;\n *Password*) echo "$COLAB_GITHUB_TOKEN" ;;\n *) exit 1 ;;\nesac\n')
askpass.chmod(0o700)
env=os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','COLAB_GITHUB_TOKEN':TOKEN})
if WORKDIR.exists():
    subprocess.run(['git','-C',str(WORKDIR),'fetch','origin',BRANCH],check=True,env=env)
    subprocess.run(['git','-C',str(WORKDIR),'checkout',BRANCH],check=True)
    subprocess.run(['git','-C',str(WORKDIR),'pull','--ff-only','origin',BRANCH],check=True,env=env)
else:
    subprocess.run(['git','clone','--single-branch','--branch',BRANCH,f'https://github.com/{REPO}.git',str(WORKDIR)],check=True,env=env)
del TOKEN,env
os.chdir(WORKDIR)
HEAD=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
print('HEAD:',HEAD)


## 3. Recuperar y verificar el artifact existente

El ZIP contiene solo referencias remotas y estadísticas; no contiene RGB ni máscaras. Se verifica el SHA-256 publicado por GitHub antes de extraerlo.


In [ ]:
ARTIFACT_ID=14770171764
RUN_ID=45038801547
EXPECTED_SHA256='b6cbcb54dfd0598e6a9011667073028be1804fc1c112d82e516a42bdbf2ba4e5'
API=f'https://api.github.com/repos/{REPO}/actions/artifacts/{ARTIFACT_ID}'
try:
    from google.colab import userdata
    api_token=userdata.get('GITHUB_TOKEN')
except Exception:
    api_token=None
if not api_token:
    api_token=getpass('GitHub PAT (Fine-grained, Actions: Read-only, acceso a este repo): ')
api_token=api_token.strip()
if not api_token:
    raise RuntimeError('No se recibió ningún token.')
headers={'Authorization':f'Bearer {api_token}','Accept':'application/vnd.github+json','X-GitHub-Api-Version':'2022-11-28'}
# Diagnosticar por separado autenticación, acceso al repositorio y permiso de Actions.
who=requests.get('https://api.github.com/user',headers=headers,timeout=30)
if who.status_code != 200:
    raise RuntimeError(f'GitHub rechazó la autenticación ({who.status_code}). Comprueba que el PAT no esté revocado/expirado y que sea un token válido. Respuesta: {who.text[:500]}')
repo_check=requests.get(f'https://api.github.com/repos/{REPO}',headers=headers,timeout=30)
if repo_check.status_code != 200:
    raise RuntimeError(f'El token autentica, pero no puede acceder al repositorio privado {REPO} ({repo_check.status_code}). En la configuración del PAT selecciona exclusivamente este repositorio y confirma acceso. Respuesta: {repo_check.text[:500]}')
meta=requests.get(API,headers=headers,timeout=60)
if meta.status_code != 200:
    body={}
    try: body=meta.json()
    except Exception: pass
    accepted=meta.headers.get('X-Accepted-GitHub-Permissions','no indicado')
    if meta.status_code == 403:
        raise RuntimeError('GitHub autentica el PAT y permite ver el repositorio, pero deniega leer Actions/artifacts (403). Solución: edita o crea un Fine-grained PAT: Resource owner = propietario del repositorio; Repository access = Only select repositories y selecciona Procesamiento-de-Im-genes-; Repository permissions > Actions = Read-only. Contents = Read-only puede mantenerse para clonar el repo. Guarda el PAT actualizado en Colab Secrets con el nombre GITHUB_TOKEN, reemplazando el anterior, y vuelve a ejecutar esta celda. Encabezado X-Accepted-GitHub-Permissions: '+accepted+'. Respuesta GitHub: '+str(body)[:700])
    if meta.status_code == 404:
        raise RuntimeError('GitHub no encuentra el artifact o el token no tiene permiso para verlo. Confirma que el PAT tiene Actions: Read-only y que el artifact 14770171764 sigue vigente. Respuesta: '+str(body)[:700])
    raise RuntimeError(f'No se pudo consultar el artifact ({meta.status_code}). X-Accepted-GitHub-Permissions={accepted}. Respuesta: {str(body)[:700]}')
meta_json=meta.json()
if meta_json.get('expired'):
    raise RuntimeError('El artifact de indexación ha expirado; no es posible descargarlo desde GitHub.')
if meta_json.get('workflow_run',{}).get('id') != RUN_ID:
    raise RuntimeError('El artifact no pertenece al run esperado 45038801547.')
r=requests.get(API+'/zip',headers=headers,timeout=300)
if r.status_code != 200:
    raise RuntimeError(f'GitHub encontró el artifact, pero falló la descarga ZIP ({r.status_code}): {r.text[:700]}')
zip_path=Path('/content/urban_annotation_artifact_14770171764.zip')
zip_path.write_bytes(r.content)
actual=hashlib.sha256(zip_path.read_bytes()).hexdigest()
print('Artifact:',meta_json.get('name'),'bytes:',zip_path.stat().st_size)
print('SHA-256:',actual)
if actual != EXPECTED_SHA256:
    raise RuntimeError('El SHA-256 no coincide con el digest verificado del artifact. No se extraerá.')
extract=Path('/content/urban_artifact_14770171764')
if extract.exists(): shutil.rmtree(extract)
extract.mkdir()
with zipfile.ZipFile(zip_path) as zf:
    for info in zf.infolist():
        target=(extract/info.filename).resolve()
        if extract.resolve() not in target.parents and target != extract.resolve():
            raise RuntimeError('Ruta insegura detectada en el ZIP.')
    zf.extractall(extract)
print('Artifact verificado y extraído:',extract)
del api_token,headers


## 4. Reconstruir los seis manifiestos y verificar los índices

El artifact original se generó sin cap (`per_class_cap=null`), por lo que los JSONL de `balanced/` conservan las 1.000/751/783 filas completas. Se usan los archivos semánticos como manifiesto base y se comprueba que las variantes de instancias tienen los mismos IDs y metadatos. Las estadísticas se contrastan fila por fila con los seis índices per-image.


In [ ]:
ART=extract
MANIFEST_DIR=Path('/content/cap200_input/manifests')
INDEX_DIR=ART/'annotation_index'
MANIFEST_DIR.mkdir(parents=True,exist_ok=True)
validation=[]
def read_jsonl(path):
    with path.open(encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]
def write_jsonl(path,rows):
    path.parent.mkdir(parents=True,exist_ok=True)
    with path.open('w',encoding='utf-8') as f:
        for row in rows: f.write(json.dumps(row,sort_keys=True,ensure_ascii=False)+'\n')
old_plan=json.loads((ART/'balanced/balanced_training_plan.json').read_text())
if old_plan.get('per_class_cap') is not None:
    raise RuntimeError('El plan de origen no está sin cap; no se puede reconstruir el universo completo.')
for ds in ('a2d2','urbansyn'):
  for split in ('train','val','test'):
    sem=read_jsonl(ART/f'balanced/{ds}_semantic_{split}.jsonl')
    ins=read_jsonl(ART/f'balanced/{ds}_instances_{split}.jsonl')
    ann=read_jsonl(INDEX_DIR/f'{ds}_{split}.annotations.jsonl')
    sem_map={str(x['sample_id']):x for x in sem}
    ins_map={str(x['sample_id']):x for x in ins}
    ann_map={str(x['sample_id']):x for x in ann}
    if len(sem_map)!=len(sem) or len(ins_map)!=len(ins) or len(ann_map)!=len(ann):
        raise RuntimeError(f'{ds}/{split}: IDs duplicados')
    if set(sem_map)!=set(ins_map) or set(sem_map)!=set(ann_map):
        raise RuntimeError(f'{ds}/{split}: IDs de manifiestos/índice no coinciden')
    for sid,row in sem_map.items():
        if any(row.get(k)!=ann_map[sid].get(k) for k in ('semantic_pixels','instance_counts')):
            raise RuntimeError(f'{ds}/{split}: estadísticas no coinciden para {sid}')
        if any(ins_map[sid].get(k)!=row.get(k) for k in ('semantic_pixels','instance_counts')):
            raise RuntimeError(f'{ds}/{split}: estadísticas semántica/instancias distintas para {sid}')
        base_sem={k:v for k,v in row.items() if k not in ('semantic_pixels','instance_counts')}
        base_ins={k:v for k,v in ins_map[sid].items() if k not in ('semantic_pixels','instance_counts')}
        if base_sem!=base_ins:
            raise RuntimeError(f'{ds}/{split}: metadatos de manifiesto distintos para {sid}')
    base=[{k:v for k,v in row.items() if k not in ('semantic_pixels','instance_counts')} for row in sem]
    # Restore the exact manifest order used by prepare_reproducible_subsets.py.
    base.sort(key=lambda row: hashlib.sha256(f"2026:{row['sample_id']}".encode('utf-8')).hexdigest())
    write_jsonl(MANIFEST_DIR/f'{ds}_{split}.jsonl',base)
    validation.append({'dataset':ds,'split':split,'manifest_rows':len(base),'index_rows':len(ann),'unique_ids':len(sem_map),'ids_match':True,'stats_match':True,'task_manifests_match':True})
if len(validation)!=6 or any(x['manifest_rows']!=x['index_rows'] for x in validation):
    raise RuntimeError('Validación de reconstrucción incompleta.')
validation_path=Path('/content/cap200_input/artifact_reconstruction_validation.json')
validation_path.parent.mkdir(parents=True,exist_ok=True)
validation_path.write_text(json.dumps({'artifact_id':ARTIFACT_ID,'source_run':RUN_ID,'checks':validation},indent=2,ensure_ascii=False)+'\n')
print(json.dumps(validation,indent=2,ensure_ascii=False))


## 5. Ejecutar el planificador real con cap 200

El límite se aplica solo a train. Validation y test permanecen completos. Se ejecuta el script del repositorio, no una implementación alternativa.


In [ ]:
!python -m pip install -q 'PyYAML>=6'
import subprocess
OUT=Path('/content/cap200_output')
if OUT.exists(): shutil.rmtree(OUT)
cmd=[sys.executable,'scripts/build_balanced_training_plan.py','--manifests-dir',str(MANIFEST_DIR),'--annotation-index-dir',str(INDEX_DIR),'--output-dir',str(OUT),'--seed','2026','--max-samples-per-split','1000','--per-class-cap','200']
proc=subprocess.run(cmd,cwd=WORKDIR,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
(OUT/'planner_stdout.log').write_text(proc.stdout,encoding='utf-8')
print(proc.stdout[-20000:])
if proc.returncode:
    raise RuntimeError(f'El planificador terminó con código {proc.returncode}; revisa planner_stdout.log')
plan=json.loads((OUT/'balanced_training_plan.json').read_text())


## 6. Auditar retención, cobertura y preservación de evaluación

También se calculan alternativas cap 300/400/500/700/1000 para valorar cuánto cambia el tamaño del entrenamiento. Son comparaciones de sensibilidad, no sustituyen la revisión metodológica.


In [ ]:
summary=[]
for ds in ('a2d2','urbansyn'):
  for split in ('train','val','test'):
    for task in ('semantic','instances'):
      x=plan['plans'][ds][split][task]
      if split=='train':
        if x['per_class_cap']!=200 or any(v>200 for v in x['selected_samples_per_class'].values()):
          raise RuntimeError(f'{ds}/{task}: cap 200 incumplido')
      else:
        if x['selected_samples']!=x['available_samples'] or x['unselected_samples']!=0:
          raise RuntimeError(f'{ds}/{split}/{task}: evaluación alterada')
      p=OUT/x['manifest']
      rows=read_jsonl(p)
      if len(rows)!=x['selected_samples'] or len({r['sample_id'] for r in rows})!=len(rows):
        raise RuntimeError(f'{ds}/{split}/{task}: manifiesto de salida inconsistente')
      source_rows=read_jsonl(MANIFEST_DIR/f'{ds}_{split}.jsonl')
      source_ids=[r['sample_id'] for r in source_rows]
      output_ids=[r['sample_id'] for r in rows]
      if split=='train' and not set(output_ids).issubset(set(source_ids)):
        raise RuntimeError(f'{ds}/{task}: train contiene IDs ajenos al manifiesto fuente')
      if split!='train' and output_ids!=source_ids:
        raise RuntimeError(f'{ds}/{split}/{task}: val/test no conserva orden e identidad exactos')
      summary.append({'dataset':ds,'split':split,'task':task,'available':x['available_samples'],'selected':x['selected_samples'],'unselected':x['unselected_samples'],'cap':x['per_class_cap'],'classes_covered':len(x['selected_samples_per_class']),'unavailable_classes':x['unavailable_classes'],'uncovered_classes':x['uncovered_classes']})
# Sensitivity analysis using the same actual CLI, without re-indexing.
comparison=[]
for cap in (300,400,500,700,1000):
  cap_out=Path(f'/content/cap_compare_{cap}')
  if cap_out.exists(): shutil.rmtree(cap_out)
  c=[sys.executable,'scripts/build_balanced_training_plan.py','--manifests-dir',str(MANIFEST_DIR),'--annotation-index-dir',str(INDEX_DIR),'--output-dir',str(cap_out),'--seed','2026','--max-samples-per-split','1000','--per-class-cap',str(cap)]
  rr=subprocess.run(c,cwd=WORKDIR,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
  if rr.returncode: raise RuntimeError(f'Falló comparación cap {cap}: {rr.stdout[-4000:]}')
  cp=json.loads((cap_out/'balanced_training_plan.json').read_text())
  comparison.append({'per_class_cap':cap,'datasets':{ds:{task:{'available':cp['plans'][ds]['train'][task]['available_samples'],'selected':cp['plans'][ds]['train'][task]['selected_samples'],'retention_pct':round(100*cp['plans'][ds]['train'][task]['selected_samples']/cp['plans'][ds]['train'][task]['available_samples'],2),'unavailable_classes':len(cp['plans'][ds]['train'][task]['unavailable_classes']),'uncovered_classes':len(cp['plans'][ds]['train'][task]['uncovered_classes'])} for task in ('semantic','instances')} for ds in ('a2d2','urbansyn')}})
(OUT/'audit_summary.json').write_text(json.dumps({'git_head':HEAD,'artifact_id':ARTIFACT_ID,'artifact_sha256':actual,'per_class_cap':200,'training_started':False,'datasets_downloaded':False,'results':summary,'cap_sensitivity':comparison},indent=2,ensure_ascii=False)+'\n')
print('RESULTADOS CAP 200')
for x in summary:
  if x['split']=='train': print(x['dataset'],x['task'],x['selected'],'/',x['available'],'retención',round(100*x['selected']/x['available'],1),'%','clases no disponibles',len(x['unavailable_classes']))
print('\nSENSIBILIDAD')
print(json.dumps(comparison,indent=2,ensure_ascii=False))


## 7. Guardar resultados en Drive

El ZIP final incluye el plan, los manifiestos por tarea/split, validación de reconstrucción y auditoría. No contiene imágenes ni máscaras. No ejecutes entrenamiento a partir de estos resultados hasta aprobar la retención y la cobertura.


In [ ]:
package=Path('/content/plan_balanceado_cap200.zip')
if package.exists(): package.unlink()
with zipfile.ZipFile(package,'w',zipfile.ZIP_DEFLATED) as zf:
  for base in (OUT,validation_path.parent):
    for p in base.rglob('*'):
      if p.is_file() and p.name!='artifact_reconstruction_validation.json':
        zf.write(p,arcname=('balanced/' if base==OUT else 'validation/')+str(p.relative_to(base)))
  zf.write(validation_path,arcname='validation/artifact_reconstruction_validation.json')
shutil.copy2(package,OUT_DRIVE/package.name)
shutil.copy2(OUT/'audit_summary.json',OUT_DRIVE/'audit_summary.json')
print('ZIP:',OUT_DRIVE/package.name,'bytes:',package.stat().st_size)
print('Resumen:',OUT_DRIVE/'audit_summary.json')
print('RUN_TRAINING sigue False; no se inició entrenamiento.')


## 8. Envíame el resultado

Comparte el bloque **RESULTADOS CAP 200** y **SENSIBILIDAD**, o el archivo `audit_summary.json`. No continúes a entrenamiento todavía. El cap 200 puede descartar una fracción importante de train; primero debemos decidir si la reducción es aceptable.


## 9. Smoke test real de integración de loaders multidataset

Esta etapa descarga únicamente una muestra de entrenamiento de A2D2 y una de UrbanSyn (RGB + etiquetas necesarias), las guarda en un directorio temporal y las elimina al terminar. Usa los índices de anotaciones existentes y la ontología versionada para comprobar los loaders semánticos y de instancias mediante `build_urban_train_loaders`. No descarga el dataset completo, no usa validación/test para pesos o sampler, no ejecuta un forward de modelo ni inicia entrenamiento. Requiere que las secciones anteriores hayan creado `MANIFEST_DIR` e `INDEX_DIR`.


In [ ]:
import importlib.util

# Dependencias remotas del smoke test: boto3 para A2D2 S3 y huggingface_hub para UrbanSyn.
missing_packages = [name for name in ("boto3", "huggingface_hub") if importlib.util.find_spec(name) is None]
if missing_packages:
    print("Instalando dependencias faltantes:", ", ".join(missing_packages))
    subprocess.run([sys.executable, "-m", "pip", "install", *missing_packages], check=True)
else:
    print("Dependencias boto3 y huggingface_hub ya disponibles.")

integration_report = OUT_DRIVE / "urban_loader_integration.json"
if not MANIFEST_DIR.is_dir() or not INDEX_DIR.is_dir():
    raise FileNotFoundError("Ejecuta primero las secciones 1–7: faltan los manifiestos reconstruidos o el índice de anotaciones verificado.")
if integration_report.exists():
    integration_report.unlink()  # evita confundir un resultado anterior con esta ejecución
command = [sys.executable, "-u", "-m", "scripts.smoke_test_urban_training_integration",
           "--manifest-dir", str(MANIFEST_DIR),
           "--annotation-index-dir", str(INDEX_DIR),
           "--output", str(integration_report), "--size", "512"]
completed = subprocess.run(command, cwd=WORKDIR, text=True, capture_output=True)
if completed.stdout:
    print("===== SALIDA DEL SMOKE TEST =====\n", completed.stdout)
if completed.returncode != 0:
    print("===== ERROR REAL (stderr) =====\n", completed.stderr or "<stderr vacío>")
    raise RuntimeError(f"Smoke test terminó con código {completed.returncode}; copia arriba la sección ERROR REAL completa.")
if completed.stderr:
    print("===== ADVERTENCIAS (stderr) =====\n", completed.stderr)
if not integration_report.is_file():
    raise FileNotFoundError("El script terminó sin error, pero no generó urban_loader_integration.json")
result = json.loads(integration_report.read_text(encoding="utf-8"))
if result.get("status") != "PASS" or result.get("training_started") is not False:
    raise RuntimeError("La prueba de integración no confirmó PASS sin entrenamiento.")
print("Informe persistido en Drive:", integration_report)
print("No se entrenó ningún modelo.")


## 10. Forward pass de modelos con muestras reales (sin entrenamiento)
Esta sección vuelve a descargar únicamente la misma muestra determinista de train por dataset y ejecuta DeepLabV3+ (20 clases) y Mask R-CNN (9 clases incluyendo background) con pesos aleatorios, sin pesos preentrenados, sin backward, optimizador ni checkpoints. Comprueba formas y valores finitos de logits/predicciones. Esto valida compatibilidad tensor-modelo, no precisión ni calidad. Requiere GPU recomendable; el script usa CPU si no hay CUDA.


In [ ]:
import json, subprocess, sys
forward_report = OUT_DRIVE / "urban_model_forward_real_samples.json"
if forward_report.exists(): forward_report.unlink()
command = [sys.executable, "-u", "-m", "scripts.smoke_test_urban_training_integration",
           "--manifest-dir", str(MANIFEST_DIR),
           "--annotation-index-dir", str(INDEX_DIR),
           "--output", str(forward_report), "--size", "512", "--model-forward"]
completed = subprocess.run(command, cwd=WORKDIR, text=True, capture_output=True)
if completed.stdout: print("===== SALIDA FORWARD REAL =====\n", completed.stdout)
if completed.returncode != 0:
    print("===== ERROR REAL (stderr) =====\n", completed.stderr or "<stderr vacío>")
    raise RuntimeError(f"Forward smoke test terminó con código {completed.returncode}.")
if completed.stderr: print("===== ADVERTENCIAS (stderr) =====\n", completed.stderr)
if not forward_report.is_file(): raise FileNotFoundError("No se generó el informe de forward real.")
result = json.loads(forward_report.read_text(encoding="utf-8"))
if result.get("status") != "PASS" or result.get("training_started") is not False or result.get("model_forward", {}).get("status") != "PASS":
    raise RuntimeError("El informe no confirma forward PASS sin entrenamiento.")
print("Informe persistido en Drive:", forward_report)
print("Forward real aprobado; no se entrenó ningún modelo.")


## 11. Prueba controlada de aprendizaje con datos reales (4 pasos, no entrenamiento completo)

Esta sección está autorizada para ejecutar exactamente cuatro actualizaciones: DeepLabV3+ con una muestra A2D2, DeepLabV3+ con una muestra UrbanSyn, Mask R-CNN con una muestra A2D2 y Mask R-CNN con una muestra UrbanSyn. Cada combinación crea un modelo nuevo con pesos aleatorios, calcula loss, ejecuta backward, comprueba gradientes finitos/no nulos y aplica un único paso SGD. Usa imágenes reales descargadas temporalmente y reducidas a 256×256 para limitar memoria. No recorre épocas, no usa val/test, no guarda checkpoints y no descarga datasets completos. No equivale a entrenamiento de producción.


In [ ]:
import json, subprocess, sys
learning_report = OUT_DRIVE / "urban_controlled_learning_step.json"
learning_stdout = OUT_DRIVE / "urban_controlled_learning_step.stdout.log"
learning_stderr = OUT_DRIVE / "urban_controlled_learning_step.stderr.log"
for path in (learning_report, learning_stdout, learning_stderr):
    if path.exists(): path.unlink()
if not MANIFEST_DIR.is_dir() or not INDEX_DIR.is_dir():
    raise FileNotFoundError("Ejecuta primero las secciones 1–7: faltan manifiestos o índices verificados.")
command = [sys.executable, "-u", "-m", "scripts.smoke_test_urban_training_integration",
           "--manifest-dir", str(MANIFEST_DIR),
           "--annotation-index-dir", str(INDEX_DIR),
           "--output", str(learning_report), "--size", "256", "--learning-step", "--learning-rate", "0.0001"]
completed = subprocess.run(command, cwd=WORKDIR, text=True, capture_output=True)
learning_stdout.write_text(completed.stdout or "", encoding="utf-8")
learning_stderr.write_text(completed.stderr or "", encoding="utf-8")
if completed.stdout: print("===== SALIDA PRUEBA DE APRENDIZAJE =====\n", completed.stdout)
if completed.returncode != 0:
    print("===== ERROR REAL (stderr) =====\n", completed.stderr or "<stderr vacío>")
    raise RuntimeError(f"La prueba controlada terminó con código {completed.returncode}. Logs guardados en Drive.")
if completed.stderr: print("===== ADVERTENCIAS (stderr) =====\n", completed.stderr)
if not learning_report.is_file(): raise FileNotFoundError("No se generó el informe de aprendizaje controlado.")
result = json.loads(learning_report.read_text(encoding="utf-8"))
learning = result.get("controlled_learning", {})
if (result.get("status") != "PASS" or result.get("training_started") is not True
    or result.get("full_training_started") is not False or learning.get("status") != "PASS"
    or learning.get("optimizer_steps_completed") != 4):
    raise RuntimeError("El JSON no confirma los cuatro pasos controlados y la ausencia de entrenamiento completo.")
print("Informe persistido en Drive:", learning_report)
print("Logs:", learning_stdout, learning_stderr)
print("Cuatro actualizaciones controladas aprobadas; entrenamiento completo no iniciado.")
